In [ ]:
import os
import sys
from pathlib import Path

ADVISOR_ROOT = Path.cwd().resolve()

os.chdir(ADVISOR_ROOT)

from agents import Module, Orchestrator, User
from recsys import create_recsys
from advisor import create_advisor
from config.llm_clients import get_recsys_configs, get_advisor_config, get_user_config
from runner import conversational_speaker_selection

In [ ]:
DOMAIN = "book"
MAX_TURNS = 10
SESSION_ID = "notebook"
N_CHOICES = 1
USER_MESSAGE = "Can you suggest two science fiction books?\n".strip()

In [ ]:
recsys_orchestrator = create_recsys(get_recsys_configs(), domain=DOMAIN)
advisor = create_advisor(
    llm_config=get_advisor_config(),
    bandit_model_path=None,
    bandit_log_path=os.path.join(str(ADVISOR_ROOT), "notebook_bandit_logs.jsonl"),
    max_turns=MAX_TURNS,
    session_id=SESSION_ID,
    ground_truth=[],
    training_phase=False,
    inject_random_gt_in_pool=False,
    inject_gt_rng=None,
    inject_gt_query="",
    inject_gt_shared_relationships=None,
    n_choices=N_CHOICES,
    domain=DOMAIN,
)
user = User(
    name="simulated_user"
)
max_round = 4 + MAX_TURNS * 2 + 2
main_module = Module(
    name="notebook_module",
    agents=[user, recsys_orchestrator, advisor],
    speaker_selection_method=conversational_speaker_selection,
    max_round=max_round,
)
orchestrator = Orchestrator(name="notebook_orchestrator", module=main_module)
user.talk_to(orchestrator, message=USER_MESSAGE)